## [STARTER] Udaplay Project

## Part 02 - Agent
In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:

1.Answer questions using internal knowledge (RAG)

2.Search the web when needed

3.Maintain conversation state

4.Return structured outputs

5.Store useful information for future use


In [78]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [79]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agent import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from dotenv import load_dotenv

In [80]:
# TODO: Load environment variables
load_dotenv(".env")
# print(os.getenv('OPENAI_API_KEY'))
# print(os.getenv('TAVILY_API_KEY'))
# print(os.getenv('api_key'))

True

## Tools
Build at least 3 tools:

    retrieve_game: To search the vector DB
    evaluate_retrieval: To assess the retrieval performance
    game_web_search: If no good, search the web


## Retrieve Game Tool

In [81]:
import os
import chromadb
from chromadb.utils import embedding_functions
from typing import List, Dict

chroma_client = chromadb.PersistentClient(path="chromadb")

embedding_fn = embedding_functions.DefaultEmbeddingFunction()

collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

result = collection.query(
    query_texts=["games between 2023 and 2024"],
    n_results=5,
    include=['metadatas', 'documents', 'distances']
)

print(result['documents'][0])


['[PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.']


In [82]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created

# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game
from lib.tooling import tool
@tool
def retrieve_game(query: str, n_results: int = 5) -> List[Dict]:
    """Semantic search: Finds most results in the vector DB.

    Args:
        query: A question about the game industry.
        n_results: Number of results to return (default: 5).

    Returns a list of results. Each element contains:
        - Platform: like Game Boy, PlayStation 5, Xbox 360...
        - Name: Name of the game
        - YearOfRelease: Year when that game was released for that platform
        - Description: Additional details about the game
    """
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["metadatas", "documents", "distances"]
    )

    output = []
    for metadata, distance in zip(results["metadatas"][0], results["distances"][0]):
        output.append({
            "Platform": metadata.get("Platform"),
            "Name": metadata.get("Name"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description"),
            "relevance_score": round(1 - distance, 3)
        })

    return output

## Evaluate Retrieval Tool

In [83]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

from typing import List, Dict
from pydantic import BaseModel, Field
from lib.llm import LLM
from lib.messages import SystemMessage, UserMessage
from lib.parsers import PydanticOutputParser


class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents are useful to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation result")


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[Dict]) -> Dict:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    Args:
        question: Original question from the user.
        retrieved_docs: Retrieved documents most similar to the user query in the Vector Database.

    The result includes:
        - useful: whether the documents are useful to answer the question
        - description: description about the evaluation result
    """
    docs_text = "\n".join(
        f"- [{i+1}] {doc.get('Name', 'Unknown')} ({doc.get('Platform')}, {doc.get('YearOfRelease')}): {doc.get('Description', '')}"
        for i, doc in enumerate(retrieved_docs)
    )

    llm = LLM()
    parser = PydanticOutputParser(model_class=EvaluationReport)

    messages = [
        SystemMessage(
            content=(
                "Your task is to evaluate if the documents are enough to respond to the query. "
                "Give a detailed explanation, so it's possible to take an action to accept it or not. "
                f"Respond ONLY with a JSON object matching this schema: {EvaluationReport.model_json_schema()}"
            )
        ),
        UserMessage(
            content=(
                f"User question: {question}\n\n"
                f"Retrieved documents:\n{docs_text}"
            )
        ),
    ]

    response = llm.invoke(messages)
    report = parser.parse(response)
    return report.model_dump()

## Game Web Search Tool

In [84]:
%pip install -q tavily-python

import os
from dotenv import load_dotenv
from tavily import TavilyClient

load_dotenv()

client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


Note: you may need to restart the kernel to use updated packages.


In [85]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry.

from datetime import datetime
from typing import Dict

from tavily import TavilyClient
import os
from lib.tooling import tool

@tool
def game_web_search(question: str) -> Dict:
    """Semantic search: Finds most results in the vector DB.

    Args:
        question: A question about the game industry.
    """
    client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

    search_result = client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )

    return {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": question
        }
    }

## Agent

In [86]:
tools = [retrieve_game, evaluate_retrieval, game_web_search]

agent = Agent(
    model_name="tavily-search",
    tools=tools,
    instructions=(
        "You are Udaplay, an AI Research Agent for the video game industry. "
        "Follow this workflow strictly when answering questions:\n"
        "1. Call 'retrieve_game' with the user's question to get a list of relevant games.\n"
        "2. Call 'evaluate_retrieval' passing BOTH the original question AND the full list of documents returned by 'retrieve_game' as 'retrieved_docs'.\n"
        "3. If 'evaluate_retrieval' returns useful=false, call 'game_web_search' with the user's question.\n"
        "4. Provide a clear, concise answer based on the information gathered.\n"
        "Always prefer the internal database. Use web search only when the retrieval evaluation says the docs are not useful."
    ),
)

In [87]:
import json
from lib.messages import BaseMessage
import copy
import re
from typing import List


def extract_urls(text):
    pattern = r'https?://[^\s\'")\]>]+'
    return re.findall(pattern, text)

def print_structured_messages(messages: List[BaseMessage]):
    """
    Prints agent conversation in a structured, human-readable format:
      User query     → the original user question
      Tools used     → ordered list of tools called (e.g. retrieve_game → evaluate_retrieval)
      Decision       → whether internal data was sufficient or web fallback was used
      Final answer   → the agent's final response text
      Source         → URL(s) from web search results, if any
    """
    user_query = None
    tools_used = []
    used_web_search = False
    final_answer = None
    sources = []
    sources_flag = False

    for m in messages:        
        if m.role == "user":
            user_query = m.content

        elif m.role == "assistant":
            if getattr(m, "tool_calls", None):
                for tc in m.tool_calls:
                    tool_name = tc.function.name
                    if tool_name not in tools_used:
                        tools_used.append(tool_name)
                    if tool_name == "game_web_search":
                        used_web_search = True
            elif m.content:
                final_answer = m.content

        elif m.role == "tool" and getattr(m, "name", None) == "game_web_search":
            try:
                url_sources = []
                result = json.loads(m.content)
                # print(f"Raw tool output for web search: type={type(m.content)}, content={result}")
                urls = extract_urls(m.content)
                for url in urls:
                    url_sources.append(url)
                sources = copy.deepcopy(url_sources)
                sources_flag = True
            except (json.JSONDecodeError, AttributeError, TypeError):
                pass

        elif m.role == "tool" and getattr(m, "name", None) != "game_web_search":            
            sources_flag = False            

    decision = (
        "Internal data insufficient, used web fallback"
        if used_web_search
        else "Internal data sufficient, no web search needed"
    )

    print(f"User query   : {user_query}")
    print(f"Tools used   : {' → '.join(tools_used) if tools_used else 'None'}")
    print(f"Decision     : {decision}")
    print(f"Final answer : {final_answer}")
    # print(f"Sources      : {sources}")
    # print(f"Source Flag      : {sources_flag}")
    if sources and sources_flag:
        print(f"Source       : {sources[0]}")
        for extra in sources[1:]:
            print(f"             : {extra}")
    print()

In [88]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

from lib.messages import BaseMessage
from lib.agent import Agent

def print_messages(messages: List[BaseMessage]):
    for m in messages: 
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

run_1 = agent.invoke(query="When Pokémon Gold and Silver was released?", session_id="pokemon")

messages = run_1.get_final_state()["messages"]
print_structured_messages(messages)


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


TypeError: LLM.__init__() got an unexpected keyword argument 'api_key'